# MIMIC-CXR 多模态胸片 + 报告解读 Notebook

本 Notebook 将现有的 `multimodal_cxr_report.py` 脚本迁移为交互式环境，方便：
- 逐步调试多模态（图像 + 文本） vs 纯文本 LLM 的解读流程；
- 按需修改提示词、模型配置和数据路径；
- 在单独单元格中查看中间结果、绘图或导出统计结果。

In [ ]:
# 1. 创建并配置 Notebook 环境单元格

import os
import sys
from pathlib import Path

# 项目根目录：根据当前 Notebook 所在位置自动推断
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent  # srtp-medical-ai
DATA_ROOT = PROJECT_ROOT.parent / "SRTP_Data"

print("当前工作目录:", CURRENT_DIR)
print("项目根目录:", PROJECT_ROOT)
print("数据目录:", DATA_ROOT)

# 确保可以 import 项目内的模块
if str(PROJECT_ROOT / "code") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "code"))


In [ ]:
# 2. 从现有 .py 文件导入代码（可选）
# 你可以直接使用 %run 运行原脚本，或者只当它是参考实现。

script_path = PROJECT_ROOT / "code" / "multimodal_cxr_report.py"
print("脚本路径:", script_path)

if script_path.exists():
    print("你可以运行下面一行来直接执行原脚本（带命令行参数）:")
    print("!python", script_path, "--help")
else:
    print("警告: 找不到原始脚本文件，请检查路径。")


In [ ]:
# 3. 将脚本中的函数和类拆分到独立单元格

import json
import re
from dataclasses import dataclass
from typing import Dict, List, Optional

import pandas as pd
from PIL import Image
from dotenv import load_dotenv

try:
    from openai import OpenAI  # type: ignore
except Exception:
    OpenAI = None  # type: ignore


# 14 类 CheXpert 标签名称（与 chexpert.csv 对应）
CHEXPERT_LABELS: List[str] = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pleural Other",
    "Support Devices",
    "No Finding",
]


@dataclass
class ModelConfig:
    """模型与路径配置（可在 Notebook 中直接修改实例字段）"""

    openai_model_multimodal: str = "gpt-4.1-mini"  # 支持 vision
    openai_model_text: str = "gpt-4.1-mini"  # 仅文本

    data_root: str = str(DATA_ROOT)
    chexpert_csv: str = str(DATA_ROOT / "mimic-cxr-2.0.0-chexpert.csv")


def parse_ids_from_path(image_path: str) -> Optional[Dict[str, str]]:
    """从 MIMIC-CXR-JPG 文件名中解析 subject_id / study_id。"""

    basename = os.path.basename(image_path)
    m = re.match(r"(\d+)_(\d+)_", basename)
    if not m:
        return None
    subject_id, study_id = m.group(1), m.group(2)
    return {"subject_id": subject_id, "study_id": study_id}


def load_chexpert_labels(config: ModelConfig) -> pd.DataFrame:
    df = pd.read_csv(config.chexpert_csv)
    missing = [c for c in CHEXPERT_LABELS if c not in df.columns]
    if missing:
        raise ValueError(f"CheXpert CSV 缺少列: {missing}")
    return df


def get_labels_for_study(df: pd.DataFrame, subject_id: str, study_id: str) -> Dict[str, Optional[float]]:
    sub_id = int(subject_id)
    stu_id = int(study_id)
    row = df[(df["subject_id"] == sub_id) & (df["study_id"] == stu_id)]
    if row.empty:
        return {}
    row = row.iloc[0]
    labels: Dict[str, Optional[float]] = {}
    for lab in CHEXPERT_LABELS:
        val = row.get(lab)
        labels[lab] = float(val) if pd.notna(val) else None
    return labels


In [ ]:
# 3-2. 模型调用与结果处理函数


def get_openai_client() -> Optional["OpenAI"]:
    """初始化 OpenAI 客户端（如果未安装 SDK 或未设置 key，则返回 None）。"""

    load_dotenv(override=False)
    api_key = os.getenv("OPENAI_API_KEY")
    if OpenAI is None or not api_key:
        return None
    return OpenAI(api_key=api_key)


def build_system_prompt() -> str:
    return (
        "你是一名放射科主治医师，需要根据胸部 X 光片和放射学报告，"
        "输出结构化的中文解读结果，用于科普给普通患者。"
        "请严格按照 JSON 格式输出，键包括: "
        "`findings`(主要影像学发现), `impression`(综合印象), "
        "`recommendations`(随访或治疗建议), `chexpert_labels`(14 类标签判断)。"
        "其中 `chexpert_labels` 是一个对象，键为 14 个疾病名，值为 'positive'、'negative' 或 'uncertain'。"
    )


def build_user_prompt_for_text(report_text: str) -> str:
    return (
        "以下是胸部 X 光的英文放射学报告全文。"
        "请忽略可能出现的患者姓名等隐私信息，只关注影像学内容。\n\n"
        f"[放射学报告]\n{report_text}\n\n"
        "请根据以上报告内容完成结构化解读，并输出 JSON。"
    )


def build_user_prompt_for_multimodal(report_text: str) -> str:
    return (
        "我会提供一张胸部 X 光图像以及对应的英文放射学报告，"
        "请你结合图像和报告一起进行判断。如果图像与报告描述不符，"
        "以图像为主，但在印象中说明差异。\n\n"
        f"[放射学报告]\n{report_text}\n\n"
        "请输出 JSON 结果。"
    )


def call_text_only_llm(report_text: str, config: ModelConfig) -> Dict:
    client = get_openai_client()
    if client is None:
        raise RuntimeError("OpenAI SDK 未安装或 OPENAI_API_KEY 未设置，无法调用文本模型。")

    sys_prompt = build_system_prompt()
    user_prompt = build_user_prompt_for_text(report_text)

    resp = client.chat.completions.create(
        model=config.openai_model_text,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )

    content = resp.choices[0].message.content
    assert content is not None
    return json.loads(content)


def call_multimodal_llm(image_path: str, report_text: str, config: ModelConfig) -> Dict:
    client = get_openai_client()
    if client is None:
        raise RuntimeError("OpenAI SDK 未安装或 OPENAI_API_KEY 未设置，无法调用多模态模型。")

    sys_prompt = build_system_prompt()
    user_prompt = build_user_prompt_for_multimodal(report_text)

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    # 在 Notebook 中可以直接使用文件 URL；如果你的接口需要 base64，可在这里改写
    image_url = "data:image/jpeg;base64,"  # 占位，实际使用时按你的 API 要求调整

    resp = client.chat.completions.create(
        model=config.openai_model_multimodal,
        messages=[
            {"role": "system", "content": sys_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )

    content = resp.choices[0].message.content
    assert content is not None
    return json.loads(content)


def compare_with_chexpert(pred_labels: Dict, gt_labels: Dict[str, Optional[float]]) -> Dict[str, Dict[str, Optional[str]]]:
    def map_gt(v: Optional[float]) -> Optional[str]:
        if v is None:
            return None
        if v > 0:
            return "positive"
        if v < 0:
            return "uncertain"
        return "negative"

    result: Dict[str, Dict[str, Optional[str]]] = {}
    for lab in CHEXPERT_LABELS:
        pred_val = None
        if isinstance(pred_labels, dict):
            raw = pred_labels.get(lab)
            if isinstance(raw, str):
                pred_val = raw.lower()
        gt_val = map_gt(gt_labels.get(lab)) if gt_labels else None
        result[lab] = {"pred": pred_val, "gt": gt_val}
    return result


def pretty_print_results(
    image_path: str,
    multimodal_res: Dict,
    text_res: Dict,
    chexpert_comparison_multi: Dict[str, Dict[str, Optional[str]]],
    chexpert_comparison_text: Dict[str, Dict[str, Optional[str]]],
) -> None:
    print("=" * 80)
    print(f"图像路径: {image_path}")
    print("=" * 80)

    print("[多模态模型（图像+报告）解读]\n")
    print("主要发现:", multimodal_res.get("findings"))
    print("综合印象:", multimodal_res.get("impression"))
    print("建议:", multimodal_res.get("recommendations"))
    print("\n与 CheXpert 标签对比 (multi):")
    for lab, vals in chexpert_comparison_multi.items():
        print(f"- {lab}: pred={vals['pred']}, gt={vals['gt']}")

    print("\n" + "-" * 80)
    print("[纯文本 LLM（仅报告）解读]\n")
    print("主要发现:", text_res.get("findings"))
    print("综合印象:", text_res.get("impression"))
    print("建议:", text_res.get("recommendations"))
    print("\n与 CheXpert 标签对比 (text-only):")
    for lab, vals in chexpert_comparison_text.items():
        print(f"- {lab}: pred={vals['pred']}, gt={vals['gt']}")


In [ ]:
# 4. 把脚本的 main 逻辑改造成按步骤运行的单元格

# 在这里配置一张测试图像和对应的报告（可以手动填写或从文件读取）

config = ModelConfig()

# 示例：手动指定一张图像路径（请根据你本机的实际文件修改）
example_image_path = DATA_ROOT / "files" / "p10" / "p10000032" / "s56699142" / "10000032_56699142_1.jpg"
print("示例图像路径:", example_image_path)

# 你可以直接在这里写入一小段英文报告，或改为从 txt 读取
example_report_text = """PA and lateral chest radiographs demonstrate hyperinflated lungs with flattening of the diaphragms. No focal consolidation or pleural effusion is identified."""

# 解析 CheXpert 标签
ids = parse_ids_from_path(str(example_image_path))
chexpert_labels: Dict[str, Optional[float]] = {}
if ids is not None:
    df_chexpert = load_chexpert_labels(config)
    chexpert_labels = get_labels_for_study(df_chexpert, ids["subject_id"], ids["study_id"])
else:
    print("警告：未能从文件名解析出 subject_id / study_id，将无法对比 CheXpert 标签。")

print("CheXpert 标签(原始数值):")
print({k: v for k, v in chexpert_labels.items() if v is not None})

In [ ]:
# 5. 在 Notebook 中进行一次多模态 vs 纯文本解读对比

# 注意：运行前需要在环境变量或 .env 中配置 OPENAI_API_KEY

# 简单验证图像能否打开
_ = Image.open(example_image_path).convert("RGB")

multimodal_res = call_multimodal_llm(str(example_image_path), example_report_text, config)
text_res = call_text_only_llm(example_report_text, config)

multi_cmp = compare_with_chexpert(multimodal_res.get("chexpert_labels", {}), chexpert_labels)
text_cmp = compare_with_chexpert(text_res.get("chexpert_labels", {}), chexpert_labels)

pretty_print_results(str(example_image_path), multimodal_res, text_res, multi_cmp, text_cmp)

In [ ]:
# 6. 简单测试与断言（可根据需要扩展）

# 示例：检查标签对比函数的输出结构
sample_pred = {lab: "positive" for lab in CHEXPERT_LABELS}
sample_gt = {lab: 1.0 for lab in CHEXPERT_LABELS}

cmp_example = compare_with_chexpert(sample_pred, sample_gt)

assert set(cmp_example.keys()) == set(CHEXPERT_LABELS)
assert all(v["pred"] == "positive" and v["gt"] == "positive" for v in cmp_example.values())

print("简单测试通过，compare_with_chexpert 输出结构正常。")

In [ ]:
# 7. （可选）将 Notebook 中的关键函数同步回 .py 文件

from textwrap import dedent

module_path = PROJECT_ROOT / "code" / "multimodal_cxr_report_from_notebook.py"

module_code = dedent('''
from dataclasses import dataclass
from typing import Dict, List, Optional
import json
import os
import re

import pandas as pd
from PIL import Image
from dotenv import load_dotenv

try:
    from openai import OpenAI  # type: ignore
except Exception:
    OpenAI = None  # type: ignore

CHEXPERT_LABELS: List[str] = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pleural Other",
    "Support Devices",
    "No Finding",
]


@dataclass
class ModelConfig:
    openai_model_multimodal: str = "gpt-4.1-mini"
    openai_model_text: str = "gpt-4.1-mini"
    data_root: str = "{data_root}"
    chexpert_csv: str = "{chexpert_csv}"


def parse_ids_from_path(image_path: str) -> Optional[Dict[str, str]]:
    basename = os.path.basename(image_path)
    m = re.match(r"(\\d+)_(\\d+)_", basename)
    if not m:
        return None
    subject_id, study_id = m.group(1), m.group(2)
    return {{"subject_id": subject_id, "study_id": study_id}}


# 其余函数可按需从 Notebook 拷贝补充...
''').format(data_root=str(DATA_ROOT), chexpert_csv=str(DATA_ROOT / "mimic-cxr-2.0.0-chexpert.csv"))

with open(module_path, "w", encoding="utf-8") as f:
    f.write(module_code)

print("已将基础骨架写入:", module_path)
print("如需完整同步，可以继续从 Notebook 复制更多函数定义。")